In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-14'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 1000

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.1111,  1.0716, -0.2187, -0.2594, -0.5264, -0.6684, -0.1796, -1.0845,
          0.3398,  0.0051, -0.6985, -0.5732]], device='cuda:0')
Scaled actions :  tensor([[-0.1111,  1.0716, -0.2187, -0.2594, -0.5264, -0.6684, -0.1796, -1.0845,
          0.3398,  0.0051, -0.6985, -0.5732]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-1.0485e-18,  1.5649e-08, -7.0849e-19,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -1.1111e-01,  1.0716e+00,
         -2.1873e-01, -2.5940e-01, -5.2638e-01, -6.6839e-01, -1.7960e-01,
         -1.0845e+00,  3.3976e-01,  5.0661e-03, -6.9848e-01, -5.7316e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1008,  1.5684, -0.3140, -0.0652, -0.1696, -0.0621, -0.2655, -1.6370,
          0.2071, -0.1074, -1.1337, -0.8943]], device='cuda:0')
Scaled actions :  tensor([[-0.1008,  1.5684, -0.3140, -0.0652, -0.1696, -0.0621, -0.2655, -1.6370,
          0.2071, -0.1074, -1.1337, -0.8943]], device='cuda:0')
obs :  tensor([[-3.4824e-02, -1.0568e-01,  4.4973e-01, -2.6035e-03,  4.2608e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.0840e-02,
          9.7257e-03, -2.9850e-03,  2.3247e-02, -7.5756e-02, -9.0533e-02,
         -4.8667e-02,  4.6749e-03,  8.9536e-03,  2.0182e-02, -9.0288e-02,
         -7.2856e-02, -1.8876e-01,  9.7551e-02, -1.1898e-02,  1.8062e-01,
         -6.8850e-01, -8.1738e-01, -3.2549e-01,  4.8177e-02,  8.6611e-02,
          1.5285e-01, -8.1601e-01, -6.4623e-01, -1.0079e-01,  1.5684e+00,
         -3.1396e-01, -6.5200e-02, -1.6958e-01, -6.2050e-02, -2.6547e-01,
         -1.6370e+00,  2.0706e-01, -1.0742e-01, -1.1337e+00, -8.9

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.0706,  1.5588,  0.2135,  0.2790, -0.1149,  0.0162, -0.0141, -1.8526,
         -0.1939, -0.5474, -0.6017, -0.5377]], device='cuda:0')
Scaled actions :  tensor([[ 0.0706,  1.5588,  0.2135,  0.2790, -0.1149,  0.0162, -0.0141, -1.8526,
         -0.1939, -0.5474, -0.6017, -0.5377]], device='cuda:0')
obs :  tensor([[-0.1484, -0.0794,  0.5476, -0.0068,  0.0044, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0603,  0.0374, -0.0060,  0.0467, -0.1578, -0.1604, -0.1269,
          0.0222,  0.0428,  0.0485, -0.3364, -0.2618, -0.0873,  0.1717, -0.0239,
          0.0502, -0.0745,  0.0828, -0.3596,  0.1093,  0.2314,  0.1076, -1.5759,
         -1.2155,  0.0706,  1.5588,  0.2135,  0.2790, -0.1149,  0.0162, -0.0141,
         -1.8526, -0.1939, -0.5474, -0.6017, -0.5377]], device='cuda:0')
torques: [  -4.00552924  200.         -200.         -200.           97.00262557
  200.           81.60513472 -200.          118.18745007 -200.
 -142.24762062 -152.31420644]
データ収集:

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.1873,  1.3842, -0.1990,  0.3238, -0.1602, -0.2389,  0.2451, -1.9170,
         -0.5006, -0.9554,  0.2184, -0.0068]], device='cuda:0')
Scaled actions :  tensor([[-0.1873,  1.3842, -0.1990,  0.3238, -0.1602, -0.2389,  0.2451, -1.9170,
         -0.5006, -0.9554,  0.2184, -0.0068]], device='cuda:0')
obs :  tensor([[-0.2326, -0.1172, -0.1994, -0.0106,  0.0125, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0317,  0.0795, -0.0072,  0.0693, -0.1478, -0.0984, -0.1344,
          0.0386,  0.0859,  0.0571, -0.6422, -0.4656,  0.2128,  0.2517,  0.0109,
          0.1609,  0.0662,  0.2489,  0.1894,  0.0527,  0.1986, -0.0149, -1.1643,
         -0.5902, -0.1873,  1.3842, -0.1990,  0.3238, -0.1602, -0.2389,  0.2451,
         -1.9170, -0.5006, -0.9554,  0.2184, -0.0068]], device='cuda:0')
torques: [  -3.66638181  200.          200.          200.            0.2432394
  -22.99891568   87.58023949 -200.         -200.         -200.
  200.          200.        ]
データ収集: 

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.1802,  1.4344,  0.4705, -0.0567, -0.2742, -0.3755, -0.2333, -1.8935,
         -0.1544, -0.6064, -0.1262, -0.1358]], device='cuda:0')
Scaled actions :  tensor([[ 0.1802,  1.4344,  0.4705, -0.0567, -0.2742, -0.3755, -0.2333, -1.8935,
         -0.1544, -0.6064, -0.1262, -0.1358]], device='cuda:0')
obs :  tensor([[-0.2828,  0.4593, -0.1302, -0.0025,  0.0229, -0.9997,  1.0000,  0.0000,
          0.0000, -0.0421,  0.1456, -0.0370,  0.1276, -0.1503, -0.1283, -0.0439,
          0.0449,  0.1018,  0.0428, -0.7674, -0.4748, -0.2302,  0.3904, -0.2492,
          0.3597, -0.0195, -0.2449,  0.5923,  0.0075, -0.0107, -0.1181, -0.1862,
          0.3970,  0.1802,  1.4344,  0.4705, -0.0567, -0.2742, -0.3755, -0.2333,
         -1.8935, -0.1544, -0.6064, -0.1262, -0.1358]], device='cuda:0')
torques: [ -91.63977455  200.          -99.22788905   51.78661873   -0.56211847
   21.97548635   14.96201184 -200.         -200.         -200.
  200.          200.        ]
データ収集:

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.5354,  1.6069,  0.5394,  0.7832, -0.3191, -0.2441, -0.8572, -1.8885,
          0.3215, -0.5323, -0.2481, -0.2056]], device='cuda:0')
Scaled actions :  tensor([[ 0.5354,  1.6069,  0.5394,  0.7832, -0.3191, -0.2441, -0.8572, -1.8885,
          0.3215, -0.5323, -0.2481, -0.2056]], device='cuda:0')
obs :  tensor([[-0.3312,  0.4835, -0.4377,  0.0158,  0.0354, -0.9992,  1.0000,  0.0000,
          0.0000, -0.0430,  0.2333, -0.0648,  0.1614, -0.1922, -0.2148,  0.0171,
          0.0465,  0.0787,  0.0346, -0.7307, -0.3588,  0.1778,  0.4688, -0.0655,
          0.0125, -0.1930, -0.3778,  0.0796,  0.0045, -0.1812, -0.0182,  0.5249,
          0.5101,  0.5354,  1.6069,  0.5394,  0.7832, -0.3191, -0.2441, -0.8572,
         -1.8885,  0.3215, -0.5323, -0.2481, -0.2056]], device='cuda:0')
torques: [ 200.          200.          200.         -200.           39.57282251
   77.76623844 -200.         -200.         -200.         -200.
  200.          -65.60682111]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.1298,  1.1364, -0.3645, -0.0671, -0.0787, -0.2298, -0.2701, -1.3916,
         -0.5014, -0.2685, -0.1847, -0.2218]], device='cuda:0')
Scaled actions :  tensor([[-0.1298,  1.1364, -0.3645, -0.0671, -0.0787, -0.2298, -0.2701, -1.3916,
         -0.5014, -0.2685, -0.1847, -0.2218]], device='cuda:0')
obs :  tensor([[-0.5003, -0.2388, -0.6060,  0.0186,  0.0529, -0.9984,  1.0000,  0.0000,
          0.0000,  0.0314,  0.3358, -0.0610,  0.1703, -0.2370, -0.2370, -0.0269,
          0.0513,  0.0682,  0.0255, -0.5956, -0.2685,  0.5220,  0.5566,  0.0820,
          0.0652, -0.2194, -0.0107, -0.4711,  0.0446,  0.0556, -0.0596,  0.6459,
          0.2553, -0.1298,  1.1364, -0.3645, -0.0671, -0.0787, -0.2298, -0.2701,
         -1.3916, -0.5014, -0.2685, -0.1847, -0.2218]], device='cuda:0')
torques: [ 200.          200.          200.          200.          200.
  -25.13213021 -200.         -200.          200.         -200.
 -200.         -200.        ]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.4490, -0.8829, -0.0618,  0.1981, -0.2854,  0.2426, -0.6304,  0.5469,
          0.3726, -0.1700, -0.0774,  0.0633]], device='cuda:0')
Scaled actions :  tensor([[ 0.4490, -0.8829, -0.0618,  0.1981, -0.2854,  0.2426, -0.6304,  0.5469,
          0.3726, -0.1700, -0.0774,  0.0633]], device='cuda:0')
obs :  tensor([[-0.5374,  0.3582, -0.2285,  0.0199,  0.0716, -0.9972,  1.0000,  0.0000,
          0.0000,  0.0918,  0.4586, -0.0589,  0.1603, -0.2016, -0.2363, -0.1173,
          0.0574,  0.0565,  0.0103, -0.3863, -0.2038,  0.1403,  0.6533, -0.0293,
         -0.1432,  0.2790,  0.0060, -0.3521,  0.0352, -0.1506, -0.1036,  1.2982,
          0.2390,  0.4490, -0.8829, -0.0618,  0.1981, -0.2854,  0.2426, -0.6304,
          0.5469,  0.3726, -0.1700, -0.0774,  0.0633]], device='cuda:0')
torques: [-200.          200.         -200.         -200.          -16.40138502
    7.22202513  200.         -200.         -200.         -200.
 -200.         -200.        ]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.2541, -4.1634, -0.3209,  0.5599,  0.7651,  0.3813,  0.4371,  3.0285,
          0.3399,  0.1845, -0.1312,  0.2862]], device='cuda:0')
Scaled actions :  tensor([[-0.2541, -4.1634, -0.3209,  0.5599,  0.7651,  0.3813,  0.4371,  3.0285,
          0.3399,  0.1845, -0.1312,  0.2862]], device='cuda:0')
obs :  tensor([[-0.6358, -0.0795, -0.2390,  0.0260,  0.0979, -0.9949,  1.0000,  0.0000,
          0.0000,  0.1700,  0.5774, -0.0772,  0.1365, -0.2166, -0.1305, -0.2421,
          0.0744,  0.0584, -0.0312, -0.0950, -0.1188,  0.5261,  0.5524, -0.1153,
         -0.0943, -0.1540,  0.8155, -0.7840,  0.1041,  0.1381, -0.2318,  1.3135,
          0.3811, -0.2541, -4.1634, -0.3209,  0.5599,  0.7651,  0.3813,  0.4371,
          3.0285,  0.3399,  0.1845, -0.1312,  0.2862]], device='cuda:0')
torques: [  98.99097372 -200.          142.23663439  200.           15.99304249
   29.46623626  -80.61711921  200.          200.          -27.66498156
 -200.          -65.69194324

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.4252, -6.1654,  0.2040,  0.7660,  1.8992,  0.9287,  0.4074,  4.1246,
         -0.1120, -0.2734, -0.8307,  0.2921]], device='cuda:0')
Scaled actions :  tensor([[-0.4252, -6.1654,  0.2040,  0.7660,  1.8992,  0.9287,  0.4074,  4.1246,
         -0.1120, -0.2734, -0.8307,  0.2921]], device='cuda:0')
obs :  tensor([[-7.0013e-01, -5.3623e-02,  6.9693e-02,  2.3312e-02,  1.2486e-01,
         -9.9190e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.4504e-01,
          6.8319e-01, -1.1669e-01,  1.4063e-01, -1.3963e-01,  5.7534e-02,
         -3.5161e-01,  1.0039e-01,  7.8677e-02, -3.6560e-02,  2.8782e-02,
          1.5770e-02,  2.3195e-01,  5.1874e-01, -2.2876e-01,  1.1043e-01,
          8.2410e-01,  7.5359e-01, -3.5845e-01,  1.5231e-01,  7.6972e-02,
          1.2259e-01,  5.0114e-05,  6.1705e-01, -4.2519e-01, -6.1654e+00,
          2.0400e-01,  7.6597e-01,  1.8992e+00,  9.2868e-01,  4.0743e-01,
          4.1246e+00, -1.1197e-01, -2.7344e-01, -8.3068e-01,  2.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 1.0096e-03, -6.0366e+00,  1.9115e-01,  1.3468e+00,  1.3720e+00,
          1.0086e+00,  1.9588e-01,  4.0504e+00,  4.0383e-01,  3.4285e-01,
         -3.1641e-01,  6.1869e-01]], device='cuda:0')
Scaled actions :  tensor([[ 1.0096e-03, -6.0366e+00,  1.9115e-01,  1.3468e+00,  1.3720e+00,
          1.0086e+00,  1.9588e-01,  4.0504e+00,  4.0383e-01,  3.4285e-01,
         -3.1641e-01,  6.1869e-01]], device='cuda:0')
obs :  tensor([[-4.6891e-01,  2.4817e-01, -2.5262e-01,  2.6949e-02,  1.4761e-01,
         -9.8868e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.1298e-01,
          7.7682e-01, -1.1385e-01,  1.7259e-01,  1.3055e-01,  2.9975e-01,
         -3.6173e-01,  1.3240e-01,  7.8124e-02, -2.5381e-02, -7.5266e-02,
          1.1233e-01, -4.9111e-01,  4.2793e-01,  2.3987e-01,  2.0242e-01,
          1.7836e+00,  1.3494e+00,  2.0478e-01,  1.7055e-01, -7.7094e-02,
         -8.4872e-04, -9.4683e-01,  3.9401e-01,  1.0096e-03, -6.0366e+00,
          1.9115e-01,  1

In [34]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=2.026, Scaled action max=2.026
Step 1/10, Total steps: 102
steps: 102
actions : tensor([[ 0.3974,  2.0264,  0.5962, -0.9736, -0.1131, -0.4973,  0.2757, -0.9182,
         -0.0385,  1.6830, -0.0498, -2.9354]], device='cuda:0')
target_dof_pos: tensor([[-0.0152,  0.3340, -0.6261, -0.9340,  0.0380, -0.0727,  1.4205, -0.2342,
         -2.9352,  1.3399,  0.8264, -2.6922]], device='cuda:0')
Step 1: Original action max=1.899, Scaled action max=1.899
Step 2: Original action max=2.462, Scaled action max=2.462
データ収集完了: 10 steps collected with action_scale=1.0


In [25]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [38]:
env.sim.stop()

In [39]:
env.reset()
cnt = 0

In [56]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-8_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (313, 58)
